# Anyone: similarity rewards cannot tell people apart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jajamoa/anyone/blob/main/notebooks/anyone.ipynb)

**Claim.** The similarity scores used to evaluate and train free-text user simulators (the HumanLM LLM judge, embedding cosine) do not respond to *which person* is being simulated. A structured probe of the person's stance and warrant does.

**Design.** 60 (user, post) pairs from r/AmItheAsshole. For each, the simulator (claude-haiku-4-5, temperature 0.1) writes the user's comment three times:

| condition | history given to the simulator |
|---|---|
| `same`  | the target's own past comments |
| `cross` | another real commenter on the **same post** who gave a different warrant |
| `same2` | the target's own past comments again: a fresh sample, the noise floor |

Every text is then scored against the real comment by HumanLM's own judge (their prompt, their model, temperature 0), by embedding cosine (all-mpnet-base-v2), and by a reader model that extracts the stance (NTA / YTA) and warrant (one of ten moral principles) from the text.

**Test.** Person effect = `same - cross`. Noise floor = `same - same2`. A metric that measures the person must show a person effect that clears its own noise floor. Confidence intervals: bootstrap over users (B = 4000, seed 13).

Every number below is computed from the frozen JSON files in `data/`. No API key is needed. The scripts that produced those files are in `src/` (see the README to rerun them).

In [1]:
import json, os, random, statistics, collections, urllib.request

SEED, B = 13, 4000
DATA = "data" if os.path.exists("data") else "../data" if os.path.exists("../data") else None
RAW = "https://raw.githubusercontent.com/jajamoa/anyone/main/data/"

def load(name):
    if DATA:
        return json.load(open(f"{DATA}/{name}.json"))
    return json.load(urllib.request.urlopen(RAW + name + ".json"))

items = load("items")          # 60 items: scenario, real comment, labels, donor
gens  = load("generations")    # simulator output under same / cross / same2, plus Method 1 answers
judge = load("judge")          # HumanLM judge score per text
emb   = load("embeddings")     # embedding cosine per text
probe = load("probe")          # reader-model stance / warrant per text (Method 2)
by_id = {it["item_id"]: it for it in items}
print(len(items), "items,", len({it["target_user"] for it in items}), "target users")

60 items, 53 target users


## 1. The data

Each item pairs a target user with a donor who commented on the same post and gave a different warrant. Histories are capped at 6000 words and never include the target post. The histories themselves are not distributed (they are other people's Reddit comment histories); the real target comments are, pseudonymized.

In [2]:
print("history words  same %.0f  cross %.0f" % (
    statistics.mean(i["ctx_same_words"] for i in items), statistics.mean(i["ctx_cross_words"] for i in items)))
print("stance labels ", dict(collections.Counter(i["stance_gt"] for i in items)))
print("warrant labels", dict(collections.Counter(i["warrant_gt"] for i in items).most_common()))
print("items per user", dict(collections.Counter(collections.Counter(i["target_user"] for i in items).values())))

history words  same 5854  cross 5857
stance labels  {'NTA': 45, 'YTA': 15}
warrant labels {'autonomy_boundaries': 18, 'property_consent': 11, 'role_obligation': 10, 'care_harm': 8, 'honesty_communication': 4, 'safety_risk': 4, 'fairness_reciprocity': 2, 'tradition_expectations': 2, 'loyalty_betrayal': 1}
items per user {2: 2, 1: 49, 3: 1, 4: 1}


## 2. One item, end to end

user_059's real comment, the three simulated comments, and what HumanLM's judge said about each. Notice that the judge's key points are the verdict and the generic advice: things every commenter on the post shares.

In [3]:
it = next(i for i in items if i["target_user"] == "user_059"); k = it["item_id"]
print("SCENARIO:", it["scenario"][:600].replace("\n", " "), "...\n")
print("REAL COMMENT (stance %s, warrant %s):\n%s\n" % (it["stance_gt"], it["warrant_gt"], it["target_comment"]))
print("JUDGE KEY POINTS:", judge[k]["same"]["key_points"], "\n")
for c in ("same", "cross", "same2"):
    print(f"--- {c}: judge {judge[k][c]['score']:.2f}, cosine {emb[k][c]:.3f}, "
          f"reader says {probe[k][c]['stance']} / {probe[k][c]['warrant']}")
    print(gens[k][c]["text"][:700], "\n")

SCENARIO: I just started a new job as a teacher straight from teacher's college and I feel like I am running on fumes. Between lesson planning, grading, emails, trying to be an inspiring teacher to students who like to chat with me during my breaks and also running the breakfast club program at school... I am feeling fatigued. My eye twitches from exhaustion and I literally fall asleep on the couch while on my laptop. I am also a mom of three and wife to a husband. I have a great relationship with my mom in law and I love her. She likes to socialize and have parties and gatherings and we are always invi ...

REAL COMMENT (stance NTA, warrant care_harm):
NTA. You need down time. It's sad you couldn't go, but your mental health and physical health matter more. If she can't understand, that's on her. I hope you're able to find a balance, though, because you need to be able to rest more and not be so run down.

JUDGE KEY POINTS: Key Point 1: Validates the user's decision (NTA - Not The Ass

## 3. Similarity metrics under the person swap

`boot` resamples users with replacement and recomputes the mean of the per-item differences, so items from the same user are never treated as independent.

In [4]:
def boot(per_item, seed=SEED):
    """per_item: {item_id: value}. Returns (mean, 2.5%, 97.5%) from a user-clustered bootstrap."""
    d = collections.defaultdict(list)
    for k, v in per_item.items():
        d[by_id[k]["target_user"]].append(v)
    users, rng = list(d), random.Random(seed)
    draws = sorted(statistics.mean(v for _ in users for v in d[rng.choice(users)]) for _ in range(B))
    return statistics.mean(per_item.values()), draws[int(.025 * B)], draws[int(.975 * B)]

def rank(a, b):
    return 1.0 if a > b else 0.0 if a < b else 0.5

def report(name, S):
    """S: {item_id: {same, cross, same2}} scores."""
    lv = {c: statistics.mean(S[k][c] for k in S) for c in ("same", "cross", "same2")}
    person = boot({k: S[k]["same"] - S[k]["cross"] for k in S})
    noise  = boot({k: S[k]["same"] - S[k]["same2"] for k in S})
    auc    = boot({k: rank(S[k]["same"], S[k]["cross"]) for k in S})
    auc2   = boot({k: rank(S[k]["same"], S[k]["same2"]) for k in S})
    print(f"{name}\n  levels        same {lv['same']:.3f}  cross {lv['cross']:.3f}  same2 {lv['same2']:.3f}")
    print(f"  person effect {person[0]:+.3f} [{person[1]:+.3f}, {person[2]:+.3f}]   (same - cross)")
    print(f"  noise floor   {noise[0]:+.3f} [{noise[1]:+.3f}, {noise[2]:+.3f}]   (same - same2)")
    print(f"  P(ranks right person above donor) {auc[0]:.3f} [{auc[1]:.3f}, {auc[2]:.3f}]"
          f"   above own resample {auc2[0]:.3f}\n")
    return lv, person, noise, auc

J = report("HumanLM judge (their prompt, claude-haiku-4-5, temperature 0)",
           {k: {c: judge[k][c]["score"] for c in judge[k]} for k in judge})
C = report("embedding cosine (all-mpnet-base-v2)", emb)

HumanLM judge (their prompt, claude-haiku-4-5, temperature 0)
  levels        same 0.516  cross 0.546  same2 0.528
  person effect -0.030 [-0.106, +0.051]   (same - cross)
  noise floor   -0.012 [-0.058, +0.038]   (same - same2)
  P(ranks right person above donor) 0.392 [0.287, 0.492]   above own resample 0.450



embedding cosine (all-mpnet-base-v2)
  levels        same 0.644  cross 0.639  same2 0.643
  person effect +0.005 [-0.011, +0.021]   (same - cross)
  noise floor   +0.000 [-0.012, +0.012]   (same - same2)
  P(ranks right person above donor) 0.483 [0.357, 0.611]   above own resample 0.467



Reading: the judge scores the *wrong* person's text higher (0.546 vs 0.516). The person effect sits inside the noise floor's interval, and the judge ranks the right person above the donor 39% of the time, below a coin flip. Embedding cosine is flat to the third decimal. Neither metric can see who is being simulated.

## 4. The structured probe under the same swap

Two ways to get the probe. **Method 1**: the simulator, given the history, answers the stance and warrant questions directly (this is what SUITE does). **Method 2**: a reader model with no history extracts stance and warrant from a free-text comment (real or generated). Accuracy is against the labels of the real comment. Warrant is 4-way, so chance is 0.25 and the majority option gives 0.30.

In [5]:
def acc(pred, lab):
    return {k: float(pred(k) == by_id[k][lab + "_gt"]) for k in by_id}

for lab in ("stance", "warrant"):
    same  = acc(lambda k: gens[k]["same"]["mcq"][lab], lab)
    cross = acc(lambda k: gens[k]["cross"]["mcq"][lab], lab)
    d = boot({k: same[k] - cross[k] for k in same})
    print(f"Method 1, {lab:<8} same {statistics.mean(same.values()):.3f}  cross {statistics.mean(cross.values()):.3f}"
          f"  person effect {d[0]:+.3f} [{d[1]:+.3f}, {d[2]:+.3f}]")
print()
for lab in ("stance", "warrant"):
    row = {c: statistics.mean(acc(lambda k: probe[k][c][lab], lab).values()) for c in ("real", "same", "cross", "same2")}
    print(f"Method 2, {lab:<8} " + "  ".join(f"{c} {v:.3f}" for c, v in row.items()))

Method 1, stance   same 0.850  cross 0.733  person effect +0.117 [+0.032, +0.212]
Method 1, warrant  same 0.350  cross 0.317  person effect +0.033 [-0.055, +0.127]

Method 2, stance   real 0.983  same 0.767  cross 0.767  same2 0.767
Method 2, warrant  real 0.667  same 0.333  cross 0.400  same2 0.400


Reading. Read off the *real* comments (Method 2, `real`), the probe recovers stance at 0.983 and warrant at 0.667 (human annotators agree with each other at about 0.73), so the probe is readable from free text. Method 1 stance moves with the person (+0.117, interval excludes zero). Method 1 warrant moves in the right direction but the interval includes zero at n = 60.

**Caveat, stated plainly.** The pilot generator (Haiku, temperature 0.1) largely ignores the history it is given: the reader finds the same stance accuracy (0.767) under `same`, `cross` and `same2`, and warrant near the majority baseline in all three. When the text does not contain the person, no ruler can find the person in it, so the structured-probe contrast in this pilot is weak. The strong version of the contrast is SUITE Table 2 (Claude Sonnet 4.6 answering the probe directly, 3 runs, many more items): warrant 50.1 under `same` versus 25.1 under `cross` (drop 25 points, run-to-run std 0.5), stance 85.8 versus 83.0. That experiment is not reproducible from this repository; it is cited, not recomputed.

What this pilot does establish on its own: with the same texts, the same swap, and HumanLM's own judge, the similarity score is flat to noise while the probe (where the generator did respond to the person, i.e. stance) is not.

## 5. Self-check against the project page

The numbers reported at https://jajamoa.github.io/anyone/ were computed from the same frozen files. This cell asserts they still hold.

In [6]:
expect = {"judge levels": ((0.516, 0.546, 0.528), tuple(round(J[0][c], 3) for c in ("same", "cross", "same2"))),
          "judge person": ((-0.030, -0.106, 0.051), tuple(round(x, 3) for x in J[1])),
          "judge noise":  ((-0.012, -0.058, 0.038), tuple(round(x, 3) for x in J[2])),
          "judge rank":   ((0.392, 0.287, 0.492), tuple(round(x, 3) for x in J[3])),
          "cosine levels": ((0.644, 0.639, 0.643), tuple(round(C[0][c], 3) for c in ("same", "cross", "same2"))),
          "cosine person": ((0.005, -0.011, 0.021), tuple(round(x, 3) for x in C[1]))}
for name, (want, got) in expect.items():
    assert want == got, (name, want, got)
print("all", len(expect), "reported numbers reproduced")

all 6 reported numbers reproduced


## 6. Why this matters for training

HumanLM and Turing-RL use this kind of similarity score as the reward in GRPO. GRPO scores G rollouts per prompt and pushes the policy toward the rollouts that beat the group mean. If the reward does not change when the person changes, the within-group ranking carries no information about the person, and the part of the policy that would model the individual is trained on sampling noise. The model learns what scores well for anyone.

## 7. Optional: recompute the embedding cosines

Everything above uses shipped numbers. This cell recomputes the embedding column from the texts (downloads a 420 MB model) and checks that it matches `data/embeddings.json`.

In [7]:
RECOMPUTE = False
if RECOMPUTE:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-mpnet-base-v2")
    worst = 0.0
    for it in items:
        k = it["item_id"]
        e = model.encode([t[:2000] for t in [it["target_comment"]] + [gens[k][c]["text"] for c in ("same", "cross", "same2")]],
                         normalize_embeddings=True)
        for i, c in enumerate(("same", "cross", "same2")):
            worst = max(worst, abs(float(e[0] @ e[i + 1]) - emb[k][c]))
    print(f"largest deviation from shipped cosines: {worst:.5f}")